# CECS 427 — Assignment 4
## Games (Chapter 6): Best Responses, Dominance, Nash Equilibrium, Mixed Strategies

### What to submit
- Completed code in the provided cells
- Payoff matrices as tables
- Best-response sets, dominant-strategy results, and equilibrium results
- Short explanations in Markdown

> Many tasks reuse the same helper functions. Implement them carefully in A1 — you’ll use them throughout.


In [18]:
# Imports
import numpy as np
import pandas as pd


### Variable Guide (quick reference)

- `actions1`, `actions2`: strategies available to Player 1 / Player 2.
- `A`: payoff matrix stored as a DataFrame where `A.loc[a1, a2] = (u1, u2)`.
- `a1`, `a2`: a specific strategy choice for Player 1 / Player 2.
- `br1`: list of best-response sets for Player 1; **br1[j]** corresponds to Player 2 action **actions2[j]**.
- `br2`: list of best-response sets for Player 2; **br2[i]** corresponds to Player 1 action **actions1[i]**.
- `dominant1`, `dominant2`: strictly dominant strategy for Player 1 / Player 2 (or `None`).
- `NE`: list of pure-strategy Nash equilibria.
- `p`, `q`: probabilities used in mixed strategies.


## Part A — Best Responses, Dominant Strategies, and a Variant Game

### A1 — Helper functions

**Goal:** Write functions that work for *any* payoff matrix.

**Input format:** a payoff matrix `A` where each entry is a tuple `(u1, u2)`  
- rows = Player 1 actions  
- columns = Player 2 actions  

**Tasks**
1. Implement `best_responses(A, actions1, actions2)`  
   - Output: **two lists**
     - `br1[j]` = list of Player 1 best responses when Player 2 plays `actions2[j]`
     - `br2[i]` = list of Player 2 best responses when Player 1 plays `actions1[i]`
2. Implement `strictly_dominant_strategy_p1(...)` and `strictly_dominant_strategy_p2(...)`.
3. Implement `pure_nash_equilibria(...)` (profiles where both are best-responding).

> You will use these functions in A2, A3, B1, B2, and B3


In [19]:
# Helper functions

def best_responses(A, actions1, actions2):
    """Compute best responses for each player.

    Parameters
    ----------
    A : pd.DataFrame
        Payoff matrix where A.loc[a1, a2] = (u1, u2)
    actions1 : list
        Player 1 actions (row labels)
    actions2 : list
        Player 2 actions (column labels)

    Returns
    -------
    br1 : list[list]
        br1[j] = list of Player 1 best responses when Player 2 plays actions2[j]
    br2 : list[list]
        br2[i] = list of Player 2 best responses when Player 1 plays actions1[i]
    """
    br1 = []
    br2 = []

    # Player 1 best responses
    for a2 in actions2:
        # Get Player 1's payoffs for Player 2's actions (u1 of the current [a1, a2])
        payoffs = [A.loc[a1, a2][0] for a1 in actions1]

        # Find max payoff
        max_payoff = max(payoffs)

        # Store all the strategies to achieve max payoff
        best_strategies = [
            a1 for a1 in actions1 if A.loc[a1, a2][0] == max_payoff
        ]
        br1.append(best_strategies)

    # Player 2 best responses
    for a1 in actions1:
        # Get Player 2's payoffs for Player1's actions (u2 of the current [a1, a2])
        payoffs = [A.loc[a1, a2][1] for a2 in actions2]

        # Find max payoff
        max_payoff = max(payoffs)

        # Store all the strategies to achieve max payoff
        best_strategies = [
            a2 for a2 in actions2 if A.loc[a1, a2][1] == max_payoff
        ]
        br2.append(best_strategies)

    return br1, br2

def strictly_dominant_strategy_p1(A, actions1, actions2):
    """Return a strictly dominant strategy for Player 1, or None."""
    br1, _ = best_responses(A, actions1, actions2)

    # If all values in br1 are the same, then we have a strictly dominant strategy.
    # Otherwise, return None
    if all(br == br1[0] for br in br1) and len(br1[0]) == 1:
        return br1[0][0]

    return None

def strictly_dominant_strategy_p2(A, actions1, actions2):
    """Return a strictly dominant strategy for Player 2, or None."""
    _, br2 = best_responses(A, actions1, actions2)

    # Same idea as above
    if all(br == br2[0] for br in br2) and len(br2[0]) == 1:
        return br2[0][0]

    return None

def pure_nash_equilibria(A, actions1, actions2):
    """Return a list of all pure-strategy Nash equilibria (a1, a2)."""
    br1, br2 = best_responses(A, actions1, actions2)
    nash_equilibria = []

    for i, a1 in enumerate(actions1):
        for j, a2, in enumerate(actions2):
            # Both players must be best-responding
            if a1 in br1[j] and a2 in br2[i]:
                nash_equilibria.append((a1, a2))

    return nash_equilibria


### A2 — 2×2 Game: Payoff construction + dominance + predicted outcome

Players: **Player 1** and **Player 2**  
Strategies for both players: `A` or `B`

The payoff is computed from two components:

**(1) Individual score (depends only on your own action)**
- If you play `A`: your individual score is 92
- If you play `B`: your individual score is 80

**(2) Shared score (depends on BOTH players’ actions)**
- If both play `B`: shared score = 96
- If exactly one player plays `B`: shared score = 92
- If neither plays `B` (i.e., both play `A`): shared score = 86

**Total payoff:** each player’s payoff is the average of  
`(their individual score)` and `(the shared score)`.

**Tasks**
1. Build the payoff matrix `A2` (2×2) using the rules above.
2. Compute `br1` and `br2` using your **A1** function.
3. Determine whether either player has a strictly dominant strategy.
4. Predict the outcome using dominance / best responses (1–3 sentences).


In [20]:
# Build payoff matrix

actions1 = ["A", "B"]
actions2 = ["A", "B"]

INDIV = {"A": 92, "B": 80}

def shared_score_A2(a1, a2):
    """Shared score for the A2 game."""
    if a1 == a2:
      if a1 == "A":
        return 86
      else:
        return 96
    else: return 92

def payoff_A2(a1, a2):
    """Return (u1,u2) for Player 1 action a1 vs Player 2 action a2."""
    shared = shared_score_A2(a1, a2)
    u1 = (INDIV[a1] + shared) / 2.0
    u2 = (INDIV[a2] + shared) / 2.0
    return (u1, u2)

A2 = pd.DataFrame(index=actions1, columns=actions2, dtype=object)

for a1 in actions1:
    for a2 in actions2:
        A2.loc[a1, a2] = payoff_A2(a1, a2)

A2


,A,B
A,"(89.0, 89.0)","(92.0, 86.0)"
B,"(86.0, 92.0)","(88.0, 88.0)"


In [21]:
# Compute & display best responses, dominant strategies
# Compute & display best responses, dominant strategies
br1, br2 = best_responses(A2, actions1, actions2)
dominant1 = strictly_dominant_strategy_p1(A2, actions1, actions2)
dominant2 = strictly_dominant_strategy_p2(A2, actions1, actions2)

print("Player 1 Best Responses (br1):")
for idx, a2 in enumerate(actions2):
    print(f"When Player 2 plays {a2}: Player 1 best response(s) = {br1[idx]}")

print("\nPlayer 2 Best Responses (br2):")
for idx, a1 in enumerate(actions1):
    print(f"When Player 1 plays {a1}: Player 2 best response(s) = {br2[idx]}")

print(f"\nStrictly Dominant Strategy Player 1: {dominant1}")
print(f"Strictly Dominant Strategy Player 2: {dominant2}")


Player 1 Best Responses (br1):
When Player 2 plays A: Player 1 best response(s) = ['A']
When Player 2 plays B: Player 1 best response(s) = ['A']

Player 2 Best Responses (br2):
When Player 1 plays A: Player 2 best response(s) = ['A']
When Player 1 plays B: Player 2 best response(s) = ['A']

Strictly Dominant Strategy Player 1: A
Strictly Dominant Strategy Player 2: A


### A2 Prediction (1–3 sentences)

For both players, action A is a strictly dominant strategy because playing A yields a higher payoff regardless of whether the opponent chooses A or B (89.0 > 86.0) when the opponent plays A, and 92.0 > 88.0 when the opponent plays B. Since both players choose their dominant strategy, the predicted outcome of the game is (A, A) with payoffs of (89.0, 89.0).

### A3 — Variant of A2 (change one parameter) + classify the game

Consider a **variant** of A2 where you change **only one** shared-score rule:

- If both play `B`: shared score becomes **100** (instead of 96)

Everything else remains identical to A2.

**Tasks**
1. Build the new payoff matrix `A3`.
2. Compute best responses, dominant strategies (if any), and the pure Nash equilibrium set.
3. Decide whether this game has the **Prisoner’s Dilemma pattern** and explain why (3–5 sentences).  
   - Definition reminder: PD has a strictly dominant “defection” strategy, but mutual “cooperation” would make both players better off than mutual defection.


In [22]:
# Build + analyze the variant game

def shared_score_A3(a1, a2):
    """Shared score for A3 variant (only BB changes)."""
    if a1 == a2:
        if a1 == "A":
            return 86
        else:  # a1 == "B"
            return 100  # Changed from 96 to 100
    else:
        return 92

def payoff_A3(a1, a2):
    shared = shared_score_A3(a1, a2)
    u1 = (INDIV[a1] + shared) / 2.0
    u2 = (INDIV[a2] + shared) / 2.0
    return (u1, u2)

A3 = pd.DataFrame(index=actions1, columns=actions2, dtype=object)
for a1 in actions1:
    for a2 in actions2:
        A3.loc[a1, a2] = payoff_A3(a1, a2)

display(A3)


,A,B
A,"(89.0, 89.0)","(92.0, 86.0)"
B,"(86.0, 92.0)","(90.0, 90.0)"


In [23]:
# Compute & display best responses, dominant strategies (if any), and pure Nash equilibrium

br1_A3, br2_A3 = best_responses(A3, actions1, actions2)
dominant1_A3 = strictly_dominant_strategy_p1(A3, actions1, actions2)
dominant2_A3 = strictly_dominant_strategy_p2(A3, actions1, actions2)
ne_A3 = pure_nash_equilibria(A3, actions1, actions2)

print(f"Player 1 Best Responses: {br1_A3}")
print(f"Player 2 Best Responses: {br2_A3}")
print(f"Strictly Dominant Strategy P1: {dominant1_A3}")
print(f"Strictly Dominant Strategy P2: {dominant2_A3}")
print(f"Pure-Strategy Nash Equilibria: {ne_A3}")

Player 1 Best Responses: [['A'], ['A']]
Player 2 Best Responses: [['A'], ['A']]
Strictly Dominant Strategy P1: A
Strictly Dominant Strategy P2: A
Pure-Strategy Nash Equilibria: [('A', 'A')]


### A3 Explanation (3–5 sentences)

* **TODO:** Explain whether the A3 variant has the Prisoner’s Dilemma pattern and why.


Yes, this variant does have the Prisoner’s Dilemma pattern. Changing the shared score to $100$ makes mutual cooperation (B, B) better for both players than mutual defection (A, A) (90.0 vs. 89.0). However, playing A remains strictly dominant for both players because defecting always gives a higher individual payout no matter what the opponent does. As a result, both players rationally choose A, leading to an inefficient Nash equilibrium at (A, A) even though (B, B) is socially superior.

---  
## Part B — Nash Equilibrium (Pure & Mixed)

### B1 — 3×3 game with a single pure Nash equilibrium (three-option game)

Players: Player 1 and Player 2  
Strategies for both: `A`, `B`, `C`

Values:
- `A` has total value 10  
- `B` has total value 4  
- `C` has total value 5  

Rules:
1. If both players choose the **same** letter:
   - they split the value equally.  
     Example: (C, C) gives each player 2.5.
2. If players choose **different** letters:
   - Player 1 earns 0
   - Player 2 earns the full value of their chosen letter
   - Special case: If Player 2 chose `A` alone (i.e., Player 1 did not choose `A`), Player 2 earns 0.

**Tasks**
1. Compute payoffs for all 9 profiles and build payoff matrix `B1`.
2. Compute best responses and check for strictly dominant strategies.
3. If dominance does not pin down the outcome, compute pure Nash equilibrium/equilibria and report the predicted outcome.


In [24]:
# Build payoff matrix

actions = ["A", "B", "C"]
values = {"A": 10, "B": 4, "C": 5}

def payoff_B1(a1, a2):
    """Return (u1,u2) using the rules in B1."""
    if a1 == a2:
        val = values[a1] / 2.0
        return (val, val)
    else:
        u1 = 0.0
        if a2 == "A":
            u2 = 0.0
        else:
            u2 = float(values[a2])
        return (u1, u2)

B1 = pd.DataFrame(index=actions, columns=actions, dtype=object)

for a1 in actions:
    for a2 in actions:
        B1.loc[a1, a2] = payoff_B1(a1, a2)
B1


,A,B,C
A,"(5.0, 5.0)","(0.0, 4.0)","(0.0, 5.0)"
B,"(0.0, 0.0)","(2.0, 2.0)","(0.0, 5.0)"
C,"(0.0, 0.0)","(0.0, 4.0)","(2.5, 2.5)"


In [25]:
# Compute & display best responses, dominance, and Nash equilibria
br1, br2 = best_responses(B1, actions, actions)
dominant1 = strictly_dominant_strategy_p1(B1, actions, actions)
dominant2 = strictly_dominant_strategy_p2(B1, actions, actions)
ne = pure_nash_equilibria(B1, actions, actions)

print("Player 1 Best Responses (br1)")
for idx, a2 in enumerate(actions):
    print(f"When Player 2 plays {a2}: Player 1 best response(s) = {br1[idx]}")

print("\nPlayer 2 Best Responses (br2)")
for idx, a1 in enumerate(actions):
    print(f"When Player 1 plays {a1}: Player 2 best response(s) = {br2[idx]}")

print(f"\nStrictly Dominant Strategy Player 1: {dominant1}")
print(f"Strictly Dominant Strategy Player 2: {dominant2}")
print(f"Pure-Strategy Nash Equilibria: {ne}")

Player 1 Best Responses (br1)
When Player 2 plays A: Player 1 best response(s) = ['A']
When Player 2 plays B: Player 1 best response(s) = ['B']
When Player 2 plays C: Player 1 best response(s) = ['C']

Player 2 Best Responses (br2)
When Player 1 plays A: Player 2 best response(s) = ['A', 'C']
When Player 1 plays B: Player 2 best response(s) = ['C']
When Player 1 plays C: Player 2 best response(s) = ['B']

Strictly Dominant Strategy Player 1: None
Strictly Dominant Strategy Player 2: None
Pure-Strategy Nash Equilibria: [('A', 'A')]


### B2 — Coordination games (multiple equilibria)

Below are **four** 2×2 games. For each game:

1. Identify which type it is:
   - Basic Coordination
   - Unbalanced Coordination
   - Battle of the Sexes
   - Stag Hunt
2. Compute best responses.
3. List all pure Nash equilibria.
4. Compare the Nash equilibria using Pareto optimality. Identify which equilibria are Pareto-optimal and explain whether Pareto optimality is sufficient to predict a unique outcome.
5. Identify the socially optimal outcome(s) by maximizing the sum of both players' payoffs. Compare the socially optimal outcome(s) with the Nash equilibria.


In [26]:
# Rows = Player 1's strategies, Columns = Player 2's strategies
import pandas as pd
strategies = ["A", "B"]

game_1 = pd.DataFrame([[(3,3),(0,0)],[(0,0),(3,3)]], index=strategies, columns=strategies)
display(game_1.style.set_caption("GAME 1"))

game_2 = pd.DataFrame([[(6,6),(0,0)],[(0,0),(2,2)]], index=strategies, columns=strategies)
display(game_2.style.set_caption("GAME 2"))

game_3 = pd.DataFrame([[(4,2),(0,0)],[(0,0),(2,4)]], index=strategies, columns=strategies)
display(game_3.style.set_caption("GAME 3"))

game_4 = pd.DataFrame([[(7,7),(0,4)],[(4,0),(3,3)]], index=strategies, columns=strategies)
display(game_4.style.set_caption("GAME 4"))

,A,B
A,"(3, 3)","(0, 0)"
B,"(0, 0)","(3, 3)"


,A,B
A,"(6, 6)","(0, 0)"
B,"(0, 0)","(2, 2)"


,A,B
A,"(4, 2)","(0, 0)"
B,"(0, 0)","(2, 4)"


,A,B
A,"(7, 7)","(0, 4)"
B,"(4, 0)","(3, 3)"


In [27]:
games = [("GAME 1", game_1), ("GAME 2", game_2), ("GAME 3", game_3), ("GAME 4", game_4)]

for name, g in games:
    br1, br2 = best_responses(g, strategies, strategies)
    ne = pure_nash_equilibria(g, strategies, strategies)
    print(f"=== {name} ===")
    print(f"Player 1 Best Responses (br1): {br1}")
    print(f"Player 2 Best Responses (br2): {br2}")
    print(f"Pure Nash Equilibria: {ne}\n")

=== GAME 1 ===
Player 1 Best Responses (br1): [['A'], ['B']]
Player 2 Best Responses (br2): [['A'], ['B']]
Pure Nash Equilibria: [('A', 'A'), ('B', 'B')]

=== GAME 2 ===
Player 1 Best Responses (br1): [['A'], ['B']]
Player 2 Best Responses (br2): [['A'], ['B']]
Pure Nash Equilibria: [('A', 'A'), ('B', 'B')]

=== GAME 3 ===
Player 1 Best Responses (br1): [['A'], ['B']]
Player 2 Best Responses (br2): [['A'], ['B']]
Pure Nash Equilibria: [('A', 'A'), ('B', 'B')]

=== GAME 4 ===
Player 1 Best Responses (br1): [['A'], ['B']]
Player 2 Best Responses (br2): [['A'], ['B']]
Pure Nash Equilibria: [('A', 'A'), ('B', 'B')]



### B2 Write-up (one short paragraph per game)

For each game, report
- the type,
- the pure Nash equilibria,
- which Nash equilibria are Pareto-optimal, and whether Pareto optimality can identify a unique outcome (explain why).
- Report the socially optimal outcome(s) and explain whether they are Nash equilibria.


#### Game 1 = Basic Coordination
- **Pure Nash equilibra** = (A, A) and (B, B)
- **Pareto-optimal** = (A, A) and (B, B). Both give the same payoff (3, 3), so Pareto optimality *cannot* identify a unique outcome.
- **Socially optimal** = (A, A) and (B, B). Both have the highest payoff 6 and are Nash equilibria.

#### Game 2 = Unbalanced Coordination
- **Pure Nash equilibra** = (A, A) and (B, B)
- **Pareto-optimal** = (A, A). Payoff (6, 6) is greater for both players than (2, 2), so Pareto optimality *can* identify a unqiue outcome.
- **Socially optimal** = (A, A). It has highest payoff of 12 and is a Nash equilibira

#### Game 3 = Battle of the Sexes
- **Pure Nash equilibra** = (A, A) and (B, B)
- **Pareto-optimal** = (A, A) and (B, B). Payoff (4, 2) favors Player 1, while (2, 4) favors Player 2. *No* unqiue outcome.
- **Socially optimal** = (A, A) and (B, B). Both have highest payoff 6 and are Nash equilibria.

#### Game 4 = Stag Hunt
- **Pure Nash equilibra** = (A, A) and (B, B)
- **Pareto-optimal** = (A, A). Payoff (7, 7) is greater for both players than (3, 3), so Pareto optimality *can* identify a unqiue outcome.
- **Socially optimal** = (A, A). It has highest payoff of 14 and is a Nash equilibira


### B3 — Mixed strategies

Given the following payoff matrix :

|     | C   | D   |
|:---:|:---:|:---:|
| **A** |  5, -5  | -1, 1  |
| **B** | -2, 2  |  4, -4  |

**Tasks**
1. Compute best responses and pure-strategy Nash equilibrium.
3. Compute the mixed-strategy equilibrium `(p, q)` using indifference.
    * Let Player 1 play `A` with probability `p` and Player 2 play `C` with probability `q`.
5. Compute the expected payoff to both players at the mixed equilibrium.


In [28]:
# Compute best responses and pure-strategy Nash equilibrium
p1_strategy = ["A", "B"]
p2_strategy = ["C", "D"]

B3 = pd.DataFrame(
    [[(5, -5), (-1, 1)],
     [(-2, 2), (4, -4)]],
    index=p1_strategy,
    columns=p2_strategy
)

display(B3)

p1_best_response, p2_best_response = best_responses(B3, p1_strategy, p2_strategy)
nash_eq = pure_nash_equilibria(B3, p1_strategy, p2_strategy)

print("Player 1 best responses by Player 2 action:")
for a2, responses in zip(p2_strategy, p1_best_response):
    print(f"  If Player 2 plays {a2}: {responses}")

print("Player 2 best responses by Player 1 action:")
for a1, responses in zip(p1_strategy, p2_best_response):
    print(f"  If Player 1 plays {a1}: {responses}")

print("Pure-strategy Nash equilibria:", nash_eq)



,C,D
A,"(5, -5)","(-1, 1)"
B,"(-2, 2)","(4, -4)"


Player 1 best responses by Player 2 action:
  If Player 2 plays C: ['A']
  If Player 2 plays D: ['B']
Player 2 best responses by Player 1 action:
  If Player 1 plays A: ['D']
  If Player 1 plays B: ['C']
Pure-strategy Nash equilibria: []


In [29]:
# Compute the mixed-strategy equilibrium
# Player 1 uses A with probability p; Player 2 uses C with probability q
# Player 1 is indifferent when: 5q + (-1)(1-q) = (-2)q + 4(1-q)
# Player 2 is indifferent when: (-5)p + 2(1-p) = 1p + (-4)(1-p)

# For Player 1: (a-c)q + (b-d)(1-q) = 0
a, b = B3.loc["A", "C"][0], B3.loc["A", "D"][0]
c, d = B3.loc["B", "C"][0], B3.loc["B", "D"][0]
q = (d - b) / (a - b - c + d)

# For Player 2: (e-f)p + (g-h)(1-p) = 0
e, f = B3.loc["A", "C"][1], B3.loc["A", "D"][1]
g, h = B3.loc["B", "C"][1], B3.loc["B", "D"][1]
p = (h - g) / (e - f - g + h)

print(f"Player 1 plays A with probability p = {p:.3f} = 1/2")
print(f"Player 1 plays B with probability 1-p = {1-p:.3f} = 1/2")
print(f"Player 2 plays C with probability q = {q:.3f} = 5/12")
print(f"Player 2 plays D with probability 1-q = {1-q:.3f} = 7/12")

Player 1 plays A with probability p = 0.500 = 1/2
Player 1 plays B with probability 1-p = 0.500 = 1/2
Player 2 plays C with probability q = 0.417 = 5/12
Player 2 plays D with probability 1-q = 0.583 = 7/12


In [30]:
# Compute the expected payoff to both players at the mixed-strategy equilibrium

# Expected payoff for each of Player 1's pure strategies
# Player 2 chooses C with probability q and D with probability 1-q
p1_if_A = 5 * q + (-1) * (1 - q)
p1_if_B = (-2) * q + 4 * (1 - q)

print(f"\nPlayer 1 expected payoff if choosing A: {p1_if_A:.3f}")
print(f"Player 1 expected payoff if choosing B: {p1_if_B:.3f}")

# Expected payoff for each of Player 2's pure strategies
# Player 1 chooses A with probability p and B with probability 1-p
p2_if_C = (-5) * p + 2 * (1 - p)
p2_if_D = 1 * p + (-4) * (1 - p)

print(f"Player 2 expected payoff if choosing C: {p2_if_C:.3f}")
print(f"Player 2 expected payoff if choosing D: {p2_if_D:.3f}")

# Mixed-strategy payoff = Weighted average of the expected payoffs for each of the player's pure strategies.
expected_payoff_p1 = p * p1_if_A + (1 - p) * p1_if_B
expected_payoff_p2 = q * p2_if_C + (1 - q) * p2_if_D

print(f"\nExpected payoff to Player 1: {expected_payoff_p1:.3f}")
print(f"Expected payoff to Player 2: {expected_payoff_p2:.3f}")



Player 1 expected payoff if choosing A: 1.500
Player 1 expected payoff if choosing B: 1.500
Player 2 expected payoff if choosing C: -1.500
Player 2 expected payoff if choosing D: -1.500

Expected payoff to Player 1: 1.500
Expected payoff to Player 2: -1.500


### Conceptual Questions (Answer in Markdown)

1. A **best response** is the best choice of one player, given a belief about what the other player will do.

2. A strategy is **strictly dominant** when it is the *strict best response* to every opponent strategy.

3. A **pure-strategy Nash equilibrium** is an outcome where neither player can improve their payoff by chanigng their strategy alone.

4. Some games require **mixed strategies** because there is no pure-strategy Nash equilibrium. Players must *randomize* among their strategies to reach an equilibirum. Yes, a game can have both pure-strategy Nash equilibrium and mixed strategy equilibrium.

5. A **strictly dominated strategy** gives a player a lower payoff than at least one alternative strategy, for every possible choice the opponent makes. Iteratively deleting the strategies removes the choices a rational player would not make, reducing the table down and making the game simpler to analyze.